# Étape 3 — Analyse des résultats (Classification Supervisée)
**Prérequis** :
1. Avoir exécuté `01_preparation_classification.ipynb` → génère `dfbase.csv`, `dfpoly.csv`, `dfinter.csv`
2. Avoir exécuté `ComparaisonCS_final.ipynb` (FICHIERS FINAUX/) **3 fois** en changeant le CSV chargé → génère `PROB.csv`, `PROBpoly.csv`, `PROBinter.csv`

Ce notebook :
- Calcule accuracy, sensibilité, spécificité, F1 pour chaque méthode
- Compare au seuil 0.5 et au **seuil naturel** (= proportion de Y=1 dans les données)
- Compare les 3 jeux de features (base, poly, interactions)
- Conclut sur la meilleure combinaison méthode × features

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import sklearn.metrics as sklm

# ← Modifier si les fichiers PROB*.csv sont ailleurs
DOSSIER_PROB = "../FICHIERS FINAUX/"

## 1. Chargement des fichiers de probabilités prédites

In [ ]:
import os

fichiers = {
    "base":   DOSSIER_PROB + "PROB.csv",
    "poly":   DOSSIER_PROB + "PROBpoly.csv",
    "inter":  DOSSIER_PROB + "PROBinter.csv"
}

probs = {}
for nom, chemin in fichiers.items():
    if os.path.exists(chemin):
        probs[nom] = pd.read_csv(chemin)
        print(f"Charge {chemin} -> {probs[nom].shape}")
    else:
        print(f"FICHIER MANQUANT : {chemin}")
        print("  Relancer ComparaisonCS_final.ipynb avec le bon dfXXX.csv et sauvegarder le bon PROB*.csv")

# Apercu d'un fichier
if "base" in probs:
    print("\nMethodes disponibles :", list(probs["base"].columns[1:]))
    print(f"Proportion Y=1 : {probs['base'].Y.mean():.1%}")
    probs["base"].head(3)

## 2. Fonction de calcul des indicateurs de performance

In [ ]:
def compute_metrics(PROB, seuil_type="fixe"):
    """Calcule les indicateurs de performance pour chaque methode.
    
    Parameters
    ----------
    PROB : DataFrame avec colonnes Y, log, BIC, AIC, ridge, lasso, elast, arbre, foret
    seuil_type : 'fixe' (s=0.5) ou 'naturel' (s = proportion de 0 dans Y)
    
    Returns
    -------
    DataFrame : une ligne par methode, colonnes = seuil, accuracy, sensibilite, specificite, medecin, F1
    """
    noms = PROB.columns[1:]   # noms des methodes (tout sauf Y)
    resultats = pd.DataFrame(index=noms)
    nbr0 = int(PROB.Y.value_counts()[0])  # nb individus Y=0 (pour seuil naturel)
    
    for nom in noms:
        if seuil_type == "naturel":
            # Seuil naturel : probabilite de coupure = proportion de Y=1 dans les donnees
            tmp = PROB[nom].sort_values(ascending=True)
            s = (tmp.iloc[nbr0-1] + tmp.iloc[nbr0]) / 2
        else:
            s = 0.5  # seuil par defaut
        
        y_pred = (PROB[nom] >= s).astype(int)
        conf = sklm.confusion_matrix(PROB.Y, y_pred)
        
        resultats.loc[nom, "seuil"]        = round(s, 3)
        resultats.loc[nom, "accuracy"]     = sklm.accuracy_score(PROB.Y, y_pred)
        resultats.loc[nom, "sensibilite"]  = conf[1,1] / (conf[1,1] + conf[1,0])  # TP / (TP+FN)
        resultats.loc[nom, "specificite"]  = conf[0,0] / (conf[0,0] + conf[0,1])  # TN / (TN+FP)
        resultats.loc[nom, "F1"]           = sklm.f1_score(PROB.Y, y_pred)
        resultats.loc[nom, "medecin"]      = resultats.loc[nom, "sensibilite"] + resultats.loc[nom, "specificite"]
    
    return resultats.astype(float).round(3)

## 3. Comparaison des methodes — seuil fixe (0.5)
Le seuil 0.5 classe un individu comme malade si sa probabilite predite depasse 50%.

In [ ]:
print("=" * 60)
for jeu, PROB in probs.items():
    print(f"\n--- Features : {jeu.upper()} ---")
    res = compute_metrics(PROB, seuil_type="fixe")
    print(res.sort_values("medecin", ascending=False).to_string())

## 4. Comparaison des methodes — seuil naturel
Le seuil naturel est calcule pour que le nombre d'individus classes positifs soit egal au nombre reel de positifs.  
C'est le seuil recommande en pratique medicale car il respecte la prevalence observee.

In [ ]:
print("=" * 60)
for jeu, PROB in probs.items():
    print(f"\n--- Features : {jeu.upper()} ---")
    res = compute_metrics(PROB, seuil_type="naturel")
    print(res.sort_values("medecin", ascending=False).to_string())

## 5. Synthese comparative — critere 'medecin' (sensibilite + specificite)
On classe toutes les combinaisons methode x features par le critere 'medecin'.

In [ ]:
# Tableau synthetique : critere 'medecin' au seuil naturel pour chaque combinaison
records = []
for jeu, PROB in probs.items():
    res = compute_metrics(PROB, seuil_type="naturel")
    for methode in res.index:
        records.append({
            "features":    jeu,
            "methode":     methode,
            "seuil":       res.loc[methode, "seuil"],
            "accuracy":    res.loc[methode, "accuracy"],
            "sensibilite": res.loc[methode, "sensibilite"],
            "specificite": res.loc[methode, "specificite"],
            "F1":          res.loc[methode, "F1"],
            "medecin":     res.loc[methode, "medecin"]
        })

synthese = pd.DataFrame(records)
print("Top 10 combinaisons par critere 'medecin' (seuil naturel) :")
print(synthese.sort_values("medecin", ascending=False).head(10).to_string(index=False))

In [ ]:
# Tableau pivot : une ligne = methode, une colonne = jeu de features
pivot = synthese.pivot(index="methode", columns="features", values="medecin")
print("Critere 'medecin' par methode et jeu de features (seuil naturel) :")
print(pivot.round(3).to_string())

## 6. Visualisation

In [ ]:
fig, axes = plt.subplots(1, len(probs), figsize=(5 * len(probs), 5), sharey=True)
if len(probs) == 1:
    axes = [axes]

for ax, (jeu, PROB) in zip(axes, probs.items()):
    res = compute_metrics(PROB, seuil_type="naturel").sort_values("medecin", ascending=True)
    ax.barh(res.index, res["medecin"], color="steelblue")
    ax.set_title(f"Features : {jeu}")
    ax.set_xlabel("Critere medecin (sens. + spec.)")
    ax.axvline(x=1.0, color="grey", linestyle="--", linewidth=0.8)  # reference : seuil aleatoire

plt.suptitle("Comparaison des methodes — seuil naturel", fontsize=13)
plt.tight_layout()
plt.show()

## 7. Conclusion
Completer apres execution :
- **Meilleure methode** : ...
- **Meilleur jeu de features** : ...
- **Critere medecin obtenu** : ...

**Pistes d'amelioration** :
- Tester d'autres valeurs de `l1_ratio` pour ElasticNet (0.25, 0.75, 0.9)
- Tester `min_samples_leaf` sur l'arbre de decision
- Tester `n_estimators` sur la foret aleatoire
- Courbes ROC pour comparaison graphique des methodes